In [5]:
# LIbrerias
!pip install pyarrow
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

In [6]:
# Ruta del archivo parquet
ruta = "data/processed/defunciones.parquet"
# 1. Cargar archivo completo
df = pd.read_parquet(ruta)

In [7]:
# 2. Filtrar años 2010–2024
df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce")

df = df[df["ANO"].between(2010, 2024)]

In [8]:
variables = [
    'ANO', 'MES', 'SEXO', 'SEXO_DESC','GRU_ED1_DESC', 'GRU_ED2', 'NIVEL_EDU', 'EST_CIVIL', 'ULTCURFAL',
    'NOMBRE_DEPARTAMENTO', 'NOMBRE_DEPARTAMENTO_OCURRENCIA','NOMBRE_DEPARTAMENTO_RESIDENCIA','NOMBRE_MUNICIPIO',
    'NOMBRE_MUNICIPIO_OCURRENCIA', 'NOMBRE_MUNICIPIO_RESIDENCIA','PMAN_MUER','PMAN_MUER_DESC','C_BAS1',  
    'C_BAS1_DESC','CONS_EXP',  'CONS_EXP_DESC','CAU_HOMOL','CAU_HOMOL_DESC','MU_PARTO', 'MU_PARTO_DESC',
    'T_PARTO', 'T_PARTO_DESC','TIPO_EMB', 'TIPO_EMB_DESC', 'T_GES',  'T_GES_DESC','T_GES_AGRU_CIE','T_GES_AGRU_CIE_DESC',
    'C_MUERTE','C_MUERTE_DESC', 'C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','ASIS_MED', 'ASIS_MED_DESC',
    'C_DIR1','C_DIR1_DESC', 'C_DIR12', 'CAUSA_666','CAUSA_666_DESC','CAUSA_667','CAUSA_667_DESC'
]
# 3. Mantener solo variables deseadas que existan en el archivo
vars_existentes = [v for v in variables if v in df.columns]

df = df[vars_existentes]

print("✅ Archivo cargado correctamente.")
print(f"✅ Registros finales: {len(df):,}")
print(f"✅ Variables cargadas: {len(vars_existentes)} / {len(variables)}")

✅ Archivo cargado correctamente.
✅ Registros finales: 3,656,068
✅ Variables cargadas: 48 / 48


In [9]:
display(df.head())

,ANO,MES,SEXO,SEXO_DESC,GRU_ED1_DESC,GRU_ED2,NIVEL_EDU,EST_CIVIL,ULTCURFAL,NOMBRE_DEPARTAMENTO,...,C_MUERTEE,ASIS_MED,ASIS_MED_DESC,C_DIR1,C_DIR1_DESC,C_DIR12,CAUSA_666,CAUSA_666_DESC,CAUSA_667,CAUSA_667_DESC
5168963,2010,03,1,MASCULINO,DE 85 A 89 AÑOS,06,13,4,0,VALLE DEL CAUCA,...,,1,SÍ,R54,SENILIDAD,,201,Tumor maligno del estómago,201,T. MALIGNO DEL ESTOMAGO
5168964,2010,03,1,MASCULINO,DE 75 A 79 AÑOS,06,02,9,99,VALLE DEL CAUCA,...,,2,NO,I219,"INFARTO AGUDO DEL MIOCARDIO, SIN OTRA ESPECIFI...",,303,Enfermedades isquémicas del corazón,303,ENFERMEDADES ISQUEMICAS DEL CORAZON
5168965,2010,03,1,MASCULINO,DE 60 A 64 AÑOS,05,02,6,5,VALLE DEL CAUCA,...,,1,SÍ,I469,"PARO CARDIACO, NO ESPECIFICADO",,611,Hiperplasia de próstata,609,"APENDICITIS, HERNIA DE LA CAVIDAD ABDOMINAL Y ..."
5168966,2010,03,2,FEMENINO,DE 65 A 69 AÑOS,06,02,6,5,VALLE DEL CAUCA,...,,1,SÍ,C780,TUMOR MALIGNO SECUNDARIO DEL PULMON,C795,202,Tumor maligno del colon y de la unión rectosig...,202,T. MALIGNO DEL COLON
5168967,2010,03,1,MASCULINO,DE 60 A 64 AÑOS,05,02,6,99,VALLE DEL CAUCA,...,,1,SÍ,R688,OTROS SINTOMAS Y SIGNOS GENERALES ESPECIFICADOS,,302,Enfermedades hipertensivas,302,ENFERMEDADES HIPERTENSIVAS


In [10]:
# COnversión de tipos de datos
cols_int = [
    "ANO", "MES", "SEXO", "NIVEL_EDU", "EST_CIVIL", "ULTCURFAL", "PMAN_MUER",
    "C_BAS1", "CONS_EXP", "CAU_HOMOL", "MU_PARTO", "T_PARTO", "TIPO_EMB",
    "T_GES", "T_GES_AGRU_CIE", "C_MUERTE", "C_MUERTEB", "C_MUERTEC",
    "C_MUERTED", "C_MUERTEE", "ASIS_MED", "C_DIR1", "C_DIR12", "CAUSA_666", "CAUSA_667"
]
for col in cols_int:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")


In [11]:
#Diagnostico Nulos
faltantes = df.isna().sum().sort_values(ascending=False)
porcentaje = (faltantes / len(df) * 100).round(2)
faltantes_df = pd.DataFrame({"Nulos": faltantes, "% del total": porcentaje})
print(faltantes_df[faltantes_df["Nulos"] > 0])


                  Nulos  % del total
C_BAS1          3656068       100.00
C_DIR1          3656067       100.00
C_DIR12         3656065       100.00
C_MUERTEE       3629972        99.29
T_GES_AGRU_CIE  3572035        97.70
T_GES           3549831        97.09
TIPO_EMB        3549830        97.09
T_PARTO         3549830        97.09
MU_PARTO        3549830        97.09
C_MUERTEC       3239260        88.60
C_MUERTE        3132303        85.67
C_MUERTED       2552967        69.83
PMAN_MUER       1739737        47.58
CAUSA_666       1739737        47.58
C_MUERTEB       1103023        30.17
CAUSA_667        402827        11.02
ULTCURFAL          3342         0.09
EST_CIVIL             1         0.00
NIVEL_EDU             1         0.00


In [12]:
df.C_BAS1_DESC.unique()

array(['', 'PERSONAS CON DESFIGURACION DEL CUELLO Y TRONCO',
       'NECESIDAD DE VACUNACION SOLO CONTRA LA TIFOIDEA-PARATIFOIDEA',
       ..., 'TUMOR MALIGNO PARAMETRIO',
       'TUMOR MALIGNO GLANDULAS PARAURETRALES',
       'ACC.FERROCARRIL COLISION OTRO OBJETO/CICLISTA'], dtype=object)

In [13]:
# Lista de columnas a eliminar
cols_drop = ["C_BAS1", "C_DIR1", "C_DIR12", "C_MUERTEE", "T_GES_AGRU_CIE", "T_GES","TIPO_EMB","T_PARTO","MU_PARTO"]

# Eliminar solo las que realmente estén en el DataFrame
df = df.drop(columns=[c for c in cols_drop if c in df.columns])

print(f"✅ Columnas eliminadas: {[c for c in cols_drop if c in df.columns]}")
print(f"📏 Nueva forma del DataFrame: {df.shape}")


✅ Columnas eliminadas: []
📏 Nueva forma del DataFrame: (3656068, 39)


In [14]:
display(df.head())

,ANO,MES,SEXO,SEXO_DESC,GRU_ED1_DESC,GRU_ED2,NIVEL_EDU,EST_CIVIL,ULTCURFAL,NOMBRE_DEPARTAMENTO,...,C_MUERTEB,C_MUERTEC,C_MUERTED,ASIS_MED,ASIS_MED_DESC,C_DIR1_DESC,CAUSA_666,CAUSA_666_DESC,CAUSA_667,CAUSA_667_DESC
5168963,2010,3,1,MASCULINO,DE 85 A 89 AÑOS,06,13,4,0,VALLE DEL CAUCA,...,1,<NA>,<NA>,1,SÍ,SENILIDAD,201,Tumor maligno del estómago,201,T. MALIGNO DEL ESTOMAGO
5168964,2010,3,1,MASCULINO,DE 75 A 79 AÑOS,06,2,9,99,VALLE DEL CAUCA,...,<NA>,<NA>,1,2,NO,"INFARTO AGUDO DEL MIOCARDIO, SIN OTRA ESPECIFI...",303,Enfermedades isquémicas del corazón,303,ENFERMEDADES ISQUEMICAS DEL CORAZON
5168965,2010,3,1,MASCULINO,DE 60 A 64 AÑOS,05,2,6,5,VALLE DEL CAUCA,...,1,<NA>,<NA>,1,SÍ,"PARO CARDIACO, NO ESPECIFICADO",611,Hiperplasia de próstata,609,"APENDICITIS, HERNIA DE LA CAVIDAD ABDOMINAL Y ..."
5168966,2010,3,2,FEMENINO,DE 65 A 69 AÑOS,06,2,6,5,VALLE DEL CAUCA,...,1,<NA>,1,1,SÍ,TUMOR MALIGNO SECUNDARIO DEL PULMON,202,Tumor maligno del colon y de la unión rectosig...,202,T. MALIGNO DEL COLON
5168967,2010,3,1,MASCULINO,DE 60 A 64 AÑOS,05,2,6,99,VALLE DEL CAUCA,...,1,<NA>,<NA>,1,SÍ,OTROS SINTOMAS Y SIGNOS GENERALES ESPECIFICADOS,302,Enfermedades hipertensivas,302,ENFERMEDADES HIPERTENSIVAS


In [15]:
df.to_parquet("C:/Users/USUARIO/Downloads/defunciones_v2.parquet", index=False)
